# Lab 3: Bottom-Up Steel Emissions with Uncertainty


---

## What this lab is

* **Part A: emissions pipeline.** Same tasks, same order, same variable names as Lab 3: operating plants 2020-2030, China utilisation by technology, plant-level production, emission factors, company emissions, projection to 2030.
* **Part B : uncertainty quantification.** You will *interrogate* your own Lab 3 numbers with the tools of the Session 4 lecture: where do the numbers come from, how sure can we be, and what would a risk committee need to know?


## Learning goals

1. Rebuild a bottom-up emissions pipeline from plant-level data (Lab 3).
2. Tell **aleatoric, epistemic and scenario** uncertainty apart.
3. Spot a **data-coverage problem** by checking against official statistics.
4. Give a **range instead of a single number** (bootstrap and simulation) and find which input matters most.
5. Compare **scenarios** and communicate the result with calibrated words.

## Conventions

* Questions are **[QUAL]** (2-4 sentences) or **[QUANT]** (a number from your code plus one sentence).
* Write each answer in the cell right under the question.
* `ttpa` = thousand tonnes per annum. Emissions in Part A are in tCO2; in Part B we switch to **MtCO2**.
* Boxes marked **[OPTIONAL]** can be skipped.
* Run **Kernel -> Restart & Run All** before submitting.

## Submission

Work in **groups of up to 4**. Fill in every member before submitting.

1. Full name: Bettelli Louis  Student ID: B00820941
2. Full name: Oscar Moulet  Student ID: B00821262
3. Full name: Juiliette Guilloux Pinard  Student ID: B00825363
4. Full name: Apolinne Seguin  Student ID: B0022617
5. Full name: Adrien Labrgere  Student ID: B00820282

**Group / repo name:** `aidams-lab3-bettelli-moulet-guilloux-pinard-seguin-labrgere`  
**Submitter (one person):** Louis Bettelli  
**Repo URL:**  https://github.com/Coachbet/aidams-lab3-bettelli-moulet-guilloux-pinard-seguin-labrgere.git

### What to submit

- This notebook (`lab3.ipynb`) with all tasks completed and cells run (outputs visible)
- A short `README.md` with how to run the notebook

Create a GitHub repo, push the notebook and README, invite the instructor, and paste the **repo URL** above.

**Send submission info to my email (1 email per group).** Include the GitHub repo URL.

**Done when:** all names are filled in and the notebook contains outputs (no need for the instructor to re-run it).


## Setup (given)

In [1]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# DATA_PATH = Path(os.environ.get("GIST_PATH", "Plant-level-data-Global-Iron-and-Steel-Tracker-September-2025-V1.xlsx"))
DATA_PATH = Path(os.environ.get("GIST_PATH", "data/Plant-level_data_Global_Iron_and_Steel_Tracker_June_2026_V1.xlsx"))
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH.resolve()}")

plant_data = pd.read_excel(DATA_PATH, sheet_name="Plant data")
capacity_data = pd.read_excel(DATA_PATH, sheet_name="Plant capacities and status")
production_data = pd.read_excel(DATA_PATH, sheet_name="Plant production")

years = list(range(2020, 2031))
SEED = 42
print(plant_data.shape, capacity_data.shape, production_data.shape)

(1293, 44) (1845, 16) (1393, 10)


In [2]:
# Match the column names used in the lab questions.
column_names = {"GEM plant ID": "Plant ID", "Country/area": "Country/Area",
                "Parent (English)": "Parent"}
for frame in (plant_data, capacity_data, production_data):
    frame.rename(columns=column_names, inplace=True)

def extract_year(values):
    return pd.to_numeric(values.astype("string").str.extract(r"(\d{4})", expand=False),
                         errors="coerce").astype("Int64")

plant_data["Start year"] = extract_year(plant_data["Start date"])
plant_data["Retired year"] = extract_year(plant_data["Retired date"])
capacity_data["Start year"] = extract_year(capacity_data["Start date"])
print("Using GIST June 2026 V1:", DATA_PATH.name)

Using GIST June 2026 V1: Plant-level_data_Global_Iron_and_Steel_Tracker_June_2026_V1.xlsx


# PART A. Emissions pipeline

## A1. Operating plants 2020-2030

**Goal.** Create `operational_plants`: one row per (plant, year) for the years in `years`, with the plant's `Country/Area`, `Status` and nominal crude steel capacity.
A plant operates in year *t* if `start year <= t <= retired year`. **Assumption (Lab 3):** a missing end date means the plant continues operating.

In [3]:
capacity_col = "Nominal crude steel capacity (ttpa)"
capacity_rows = capacity_data[["Plant ID", "Status", "Start year", capacity_col]].merge(
    plant_data[["Plant ID", "Country/Area", "Owner", "Parent",
                "Main production equipment", "Start year", "Retired year"]],
    on="Plant ID", how="left", suffixes=("_capacity", "_plant")
)
capacity_rows["Start year"] = capacity_rows["Start year_capacity"].fillna(
    capacity_rows["Start year_plant"]
)
capacity_rows[capacity_col] = pd.to_numeric(
    capacity_rows[capacity_col], errors="coerce"
).fillna(0)

# Keep capacity rows for the status check, then make one row per plant-year.
operational_capacity_rows = pd.concat(
    [capacity_rows.assign(Year=year) for year in years], ignore_index=True
)
operational_capacity_rows = operational_capacity_rows[
    (operational_capacity_rows["Start year"] <= operational_capacity_rows["Year"])
    & (operational_capacity_rows["Retired year"].isna()
       | (operational_capacity_rows["Year"] <= operational_capacity_rows["Retired year"]))
]
operational_plants = operational_capacity_rows.groupby(
    ["Plant ID", "Year"], as_index=False
).agg({
    "Country/Area": "first", "Owner": "first", "Parent": "first",
    "Main production equipment": "first",
    "Status": lambda x: "; ".join(sorted(set(x))),
    capacity_col: "sum"
})
print(operational_plants.shape)


(11514, 8)


### Q1 [QUANT] Is every "operating" plant really operating?

Using your `operational_plants` for **China**: for the years **2020 and 2026**, how many plants and how much capacity (ttpa) are counted as operating although their **`Status`** is not an operating status (look at the values of `Status` first)?
Give the share of the Chinese total.

In [4]:
print("Status values:", sorted(capacity_data["Status"].dropna().unique()))
operating_statuses = ["operating", "operating pre-retirement"]
for year in [2020, 2026]:
    china_rows = operational_capacity_rows[
        (operational_capacity_rows["Country/Area"] == "China")
        & (operational_capacity_rows["Year"] == year)
    ]
    non_operating = china_rows[~china_rows["Status"].isin(operating_statuses)]
    china_total = operational_plants[
        (operational_plants["Country/Area"] == "China")
        & (operational_plants["Year"] == year)
    ]
    print(year, "plants:", non_operating["Plant ID"].nunique(),
          "/", len(china_total),
          "capacity:", round(non_operating[capacity_col].sum()),
          "/", round(china_total[capacity_col].sum()))


Status values: ['announced', 'cancelled', 'construction', 'mothballed', 'mothballed pre-retirement', 'operating', 'operating pre-retirement', 'retired']
2020 plants: 207 / 412 capacity: 386464 / 1409754
2026 plants: 161 / 380 capacity: 272990 / 1359674


**Your answer (Q1):**

In 2020, 207 of 412 Chinese plants (50.2%) had at least one non-operating status row counted by the date rule. These rows represented 386,464 of 1,409,754 ttpa (27.4%) of capacity. In 2026, the figures were 161 of 380 plants (42.4%) and 272,990 of 1,359,674 ttpa (20.1%). This shows that dates alone do not confirm that capacity is actually operating.

## A2. China production and capacity

**Goal.** Create
* `crude_steel`: plant-year production (crude steel only), with `Country/Area`, `Owner`, `Parent` and `Main production equipment` attached;
* `china_production`: China total production by year (a Series indexed by year);
* `china_operational`: the rows of `operational_plants` for China;
* `china_capacity`: China total capacity by year.

In [5]:
production_years = [year for year in years if year in production_data.columns]
crude_steel = production_data[
    production_data["Type of production"] == "Crude steel production (ttpa)"
].melt(id_vars="Plant ID", value_vars=production_years,
       var_name="Year", value_name="Production_ttpa")
crude_steel["Production_ttpa"] = pd.to_numeric(
    crude_steel["Production_ttpa"], errors="coerce"
)
crude_steel = crude_steel.merge(
    plant_data[["Plant ID", "Country/Area", "Owner", "Parent",
                "Main production equipment"]],
    on="Plant ID", how="left"
)
china_production = crude_steel[
    crude_steel["Country/Area"] == "China"
].groupby("Year")["Production_ttpa"].sum(min_count=1)
china_operational = operational_plants[
    operational_plants["Country/Area"] == "China"
].copy()
china_capacity = china_operational.groupby("Year")[capacity_col].sum()
print(china_production)


Year
2020   179,824.00
2021    54,817.00
2022   122,752.00
2023    97,068.00
2024    77,204.00
2025          NaN
Name: Production_ttpa, dtype: float64


## A3. Utilisation by technology

**Goal.** Compute the utilisation rate (production / capacity) separately for **BF-BOF** and **EAF** (and a residual class "Other").
Create `classify_technology` (function), `china_operational` with a `Technology_type` column, `china_capacity_by_tech`, `china_production_by_tech`, `utilization_by_tech` and `utilization_by_tech_filled`.

In [6]:
def classify_technology(equipment):
    equipment = str(equipment)
    if "BF" in equipment and "BOF" in equipment:
        return "BF-BOF"
    if "EAF" in equipment:
        return "EAF"
    return "Other"

china_operational["Technology_type"] = china_operational[
    "Main production equipment"
].map(classify_technology)
crude_steel["Technology_type"] = crude_steel[
    "Main production equipment"
].map(classify_technology)
china_capacity_by_tech = china_operational.groupby(
    ["Year", "Technology_type"]
)[capacity_col].sum().unstack()
china_production_by_tech = crude_steel[
    crude_steel["Country/Area"] == "China"
].groupby(["Year", "Technology_type"])["Production_ttpa"].sum(
    min_count=1
).unstack()
utilization_by_tech = china_production_by_tech / china_capacity_by_tech
utilization_by_tech_filled = utilization_by_tech.reindex(years).ffill().bfill()
utilization_by_tech_filled["Other"] = utilization_by_tech_filled["Other"].fillna(
    (china_production / china_capacity).reindex(years).ffill().bfill()
)
print(utilization_by_tech.round(3))


Technology_type  BF-BOF  EAF  Other
Year                               
2020               0.14 0.02    NaN
2021               0.04 0.01    NaN
2022               0.09 0.04    NaN
2023               0.08 0.04    NaN
2024               0.06 0.01    NaN
2025                NaN  NaN    NaN
2026                NaN  NaN    NaN
2027                NaN  NaN    NaN
2028                NaN  NaN    NaN
2029                NaN  NaN    NaN
2030                NaN  NaN    NaN


## A4. Plant-level production

**Goal.** Attribute each technology's utilisation rate to its plants: add `Utilization_rate_tech` and `Estimated_production_ttpa` (= capacity x utilisation) to `china_operational`.

In [7]:
china_operational["Utilization_rate_tech"] = [
    utilization_by_tech_filled.loc[year, technology]
    for year, technology in zip(
        china_operational["Year"], china_operational["Technology_type"]
    )
]
china_operational["Estimated_production_ttpa"] = (
    china_operational[capacity_col] * china_operational["Utilization_rate_tech"]
)
print(china_operational[["Year", "Plant ID", "Technology_type",
                         "Estimated_production_ttpa"]].head())


     Year       Plant ID Technology_type  Estimated_production_ttpa
678  2020  P100000120068          BF-BOF                     282.37
679  2021  P100000120068          BF-BOF                      85.25
680  2022  P100000120068          BF-BOF                     188.93
681  2023  P100000120068          BF-BOF                     151.66
682  2024  P100000120068          BF-BOF                     125.82


## A5. Emission factors and plant emissions

**Goal.** Create `emission_factors` (dict), `map_emission_factor` (function), and add `Emission_factor_tCO2_per_t` and `Estimated_emissions_tCO2` to `china_operational`.

Use these emissions factors (tCO2 per tonne crude steel, after Hasanbeigi 2019), keyed by the **exact** equipment string, with **1.6** for anything else:

| `BF; BOF` 2.1 | `BF; BOF; EAF` 1.9 | `BF; BOF; EAF; DRI` 1.9 | `BF; BOF; DRI` 2.0 | `BF; EAF` 1.9 |
|---|---|---|---|---|
| `BF` 2.2 | `EAF` 0.4 | `EAF; DRI` 1.2 | `DRI` 1.3 | `Steel other/unspecified` 1.6 |

> **Hint.** Emissions (t) = production (ttpa) x 1,000 x factor. `Series.map` with a function, or `dict.get(key, 1.6)`.

In [8]:
emission_factors = {
    "BF; BOF": 2.1, "BF; BOF; EAF": 1.9,
    "BF; BOF; EAF; DRI": 1.9, "BF; BOF; DRI": 2.0,
    "BF; EAF": 1.9, "BF": 2.2, "EAF": 0.4,
    "EAF; DRI": 1.2, "DRI": 1.3,
    "Steel other/unspecified": 1.6
}

def map_emission_factor(equipment):
    return emission_factors.get(equipment, 1.6)

china_operational["Emission_factor_tCO2_per_t"] = china_operational[
    "Main production equipment"
].map(map_emission_factor)
china_operational["Estimated_emissions_tCO2"] = (
    china_operational["Estimated_production_ttpa"] * 1000
    * china_operational["Emission_factor_tCO2_per_t"]
)
print(china_operational.groupby("Year")["Estimated_emissions_tCO2"].sum().tail())


Year
2026   150,767,674.05
2027   150,597,024.75
2028   150,597,024.75
2029   150,597,024.75
2030   150,597,024.75
Name: Estimated_emissions_tCO2, dtype: float64


## A6. Company emissions

**Goal.** Create `Company` (= `Parent`, or `Owner` when `Parent` is missing), `company_emissions` (columns `Year`, `Company`, `estimated_emissions_tCO2`) and `top_companies_latest` (top 10 in the **latest year**).

In [9]:
china_operational["Company"] = china_operational["Parent"].replace(
    "unknown", pd.NA
).fillna(china_operational["Owner"])
company_emissions = china_operational.groupby(
    ["Year", "Company"], as_index=False
)["Estimated_emissions_tCO2"].sum().rename(
    columns={"Estimated_emissions_tCO2": "estimated_emissions_tCO2"}
)
top_companies_latest = company_emissions[
    company_emissions["Year"] == company_emissions["Year"].max()
].nlargest(10, "estimated_emissions_tCO2")
print(top_companies_latest.to_string(index=False))


 Year                                                               Company  estimated_emissions_tCO2
 2030                                          Angang Steel Co Ltd [100.0%]              4,300,676.84
 2030                                  Baoshan Iron & Steel Co Ltd [100.0%]              4,094,620.57
 2030                                      Beijing Shougang Co Ltd [100.0%]              2,957,501.73
 2030           Jinghua Rigang Holding Group Co Ltd [98.8%]; unknown [1.2%]              2,937,998.96
 2030                                 Maanshan Iron & Steel Co Ltd [100.0%]              2,486,289.48
 2030                          Jiangsu Shagang Iron & Steel Co Ltd [100.0%]              2,460,589.85
 2030                     Inner Mongolia BaoTou Steel Union Co Ltd [100.0%]              2,339,106.43
 2030                                          Jingye Group Co Ltd [100.0%]              2,326,461.08
 2030                       Guangxi Shenglong Metallurgical Co Ltd [100.0%]       

### Q2 [QUAL] Look at the company names

Read the names in `top_companies_latest`. Give **two data-quality problems** you notice, and explain how each could bias a *financed-emissions* figure for a lender (hint: ownership shares, joint ventures).

**Your answer (Q2):**

Some Parent entries combine several owners and percentages in one name, such as Jinghua Rigang Holding Group plus an unknown 1.2% owner. Treating that whole string as one company can assign too much emissions to a lender's borrower. Other entries have a much larger unknown stake, such as 40.8% for British Asia Pan Ruogen, so part of the emissions cannot be attributed reliably.

## A7. Projection to 2030

**Goal.** Create `actual_years` (years with production data), `forecast_years` (the years after the last observed one), `projected_company_emissions` (company emissions in the forecast years) and a function `top_company_emissions(df, year)` returning the top 5 companies of that year. Print the top 5 for the first three forecast years.

In [10]:
actual_years = china_production.dropna().index.tolist()
forecast_years = [year for year in years if year > max(actual_years)]
projected_company_emissions = company_emissions[
    company_emissions["Year"].isin(forecast_years)
].copy()

def top_company_emissions(df, year):
    return df[df["Year"] == year].nlargest(5, "estimated_emissions_tCO2")

for year in forecast_years[:3]:
    print("\n", year)
    print(top_company_emissions(projected_company_emissions, year).to_string(index=False))



 2025
 Year                                                     Company  estimated_emissions_tCO2
 2025                                Angang Steel Co Ltd [100.0%]              4,300,676.84
 2025                        Baoshan Iron & Steel Co Ltd [100.0%]              4,094,620.57
 2025 Jinghua Rigang Holding Group Co Ltd [98.8%]; unknown [1.2%]              2,937,998.96
 2025                            Beijing Shougang Co Ltd [100.0%]              2,802,108.64
 2025                       Maanshan Iron & Steel Co Ltd [100.0%]              2,486,289.48

 2026
 Year                                                     Company  estimated_emissions_tCO2
 2026                                Angang Steel Co Ltd [100.0%]              4,300,676.84
 2026                        Baoshan Iron & Steel Co Ltd [100.0%]              4,094,620.57
 2026                            Beijing Shougang Co Ltd [100.0%]              2,957,501.73
 2026 Jinghua Rigang Holding Group Co Ltd [98.8%]; unknown [1.2%] 

### Q3 [QUAL] Forecast, projection or scenario?

The top-5 tables for the first projected years are (almost) identical. (a) Why? (b) Is this a **forecast**, a **projection** or a **scenario**, and what is it conditional on? (c) Name **three things** a fan chart would tell a risk manager that this table does not.

**Your answer (Q3):**

The tables barely change because I carry the last observed utilisation rate forward and change only the capacity implied by the plant dates. This is a projection conditional on those assumptions, not a prediction of future demand. A fan chart would show the range of possible emissions, how that range changes over time, and whether company ranges overlap.

---
# PART B. How much can we trust these numbers?

You now have company emissions as **single numbers**. Part B asks three simple questions:

1. **Do the numbers agree with an outside reference?** (reality check)
2. **How wide is the range of plausible values?** (an interval, found by simulation)
3. **What does the range depend on?** (inputs and scenarios)

**Three words you need**

| Word | Meaning | Example |
|---|---|---|
| **Aleatoric** uncertainty | pure randomness, more data does *not* remove it | will a given year be a strong or a weak year for steel? |
| **Epistemic** uncertainty | we lack knowledge, more data or better models *can* reduce it | what is the true average emission factor of a blast furnace? |
| **Scenario** uncertainty | depends on future choices, it is not a probability | will China retire old blast furnaces by 2030? |

A **90% interval** is a range that contains the true value in 90% of cases *if our assumptions are right*.

## B0. Think first

> **Pause & think.** No code.

### Q4 [QUAL] Which kind of uncertainty?

For each input of your pipeline, write **A** (aleatoric), **E** (epistemic) or **S** (scenario), and say whether **more data could reduce it** (yes / no). One short line each.

| Input | A / E / S | More data helps? |
|---|---|---|
| Utilisation rate of a blast furnace next year | | |
| Emission factor of BF-BOF steel | | |
| Share of plants that report their production | | |
| Steel demand in 2030 | | |

**Your answer (Q4):**

| Input | Type | More data helps? |
|---|---|---|
| Blast-furnace utilisation next year | A | No: the year's random variation remains. |
| BF-BOF emission factor | E | Yes: direct measurements could improve it. |
| Share of plants reporting production | E | Yes: better reporting would reveal it. |
| Steel demand in 2030 | S | No: it depends on future choices and conditions. |

## B1. Reality check against official statistics

Official crude steel production of China (**rounded worldsteel figures: verify them and cite your source**):

| Year | 2020 | 2021 | 2022 | 2023 |
|---|---|---|---|---|
| Mt | 1,065 | 1,035 | 1,015 | 1,020 |

**Goal.** Create `official_kt` (dict, in **ttpa**: 1 Mt = 1,000 ttpa) and a DataFrame `check` with, for 2020-2023: the production in your dataset (`china_production`), the official value, the **coverage** (dataset / official) and the **implied utilisation** (official / `china_capacity`).

> **Hint.** `pd.Series(official_kt)`, `pd.DataFrame({...})`, `.dropna()`.

In [11]:
# Rounded reference figures provided in the exercise (Mt converted to ttpa).
official_kt = {2020: 1065000, 2021: 1035000, 2022: 1015000, 2023: 1020000}
check = pd.DataFrame({
    "dataset_ttpa": china_production,
    "official_ttpa": pd.Series(official_kt),
    "capacity_ttpa": china_capacity
}).loc[2020:2023].dropna()
check["coverage"] = check["dataset_ttpa"] / check["official_ttpa"]
check["implied_utilization"] = check["official_ttpa"] / check["capacity_ttpa"]
print(check.round(3))


      dataset_ttpa  official_ttpa  capacity_ttpa  coverage  \
2020    179,824.00   1,065,000.00   1,409,754.00      0.17   
2021     54,817.00   1,035,000.00   1,412,457.00      0.05   
2022    122,752.00   1,015,000.00   1,398,441.00      0.12   
2023     97,068.00   1,020,000.00   1,366,241.00      0.10   

      implied_utilization  
2020                 0.76  
2021                 0.73  
2022                 0.73  
2023                 0.75  


### Q5 [QUANT + QUAL] What does your pipeline miss?

(a) Give the **coverage** and **implied utilisation** for 2023, and the factor by which your Part A pipeline **under-estimates** China's production. (b) The BF-BOF "utilisation" in Part A drops from about 57% to about 8% between 2020 and 2021. Is that a real change in how furnaces are run, or something else? Which kind of uncertainty is it (A, E or S)?

**Your answer (Q5):**

In 2023, dataset coverage is **9.5%** and official-implied utilisation is **74.7%**. Part A estimates about 97,085 ttpa against 1,020,000 ttpa officially, an underestimate by **10.5 times**. In this June 2026 dataset, naive BF-BOF utilisation falls from **14.1% in 2020 to 4.3% in 2021**; this is mainly a reporting-coverage problem, so it is epistemic uncertainty.

I used the exercise's rounded official figures. [Worldsteel's 2021 totals](https://worldsteel.org/media/press-releases/2022/december-2021-crude-steel-production-and-2021-global-totals/) and [World Steel in Figures 2024](https://worldsteel.org/data/world-steel-in-figures/world-steel-in-figures-2024/) give slightly different revised values (for example, 1,019.1 Mt for 2022 and 2023).

## B2. Fix the denominator: compare like with like

Use only plants that **report production** *and* have a capacity ("matched" plants), so that numerator and denominator describe the same plants.

**Goal.** Create `obs_matched` (one row per plant-year with production **and** capacity > 0, with `Technology_type` and `u` = production / capacity) and `utilization_matched` (year x technology: **sum of production / sum of capacity** of the matched plants).

> **Hint.** Merge China's `crude_steel` rows with `china_operational[["Plant ID", "Year", capacity]]` on `Plant ID` and `Year`.

In [12]:
obs_matched = crude_steel[
    (crude_steel["Country/Area"] == "China")
    & crude_steel["Production_ttpa"].notna()
].merge(
    china_operational[["Plant ID", "Year", capacity_col]],
    on=["Plant ID", "Year"], how="inner"
)
obs_matched = obs_matched[obs_matched[capacity_col] > 0].copy()
obs_matched["u"] = obs_matched["Production_ttpa"] / obs_matched[capacity_col]
matched_totals = obs_matched.groupby(
    ["Year", "Technology_type"]
)[["Production_ttpa", capacity_col]].sum()
utilization_matched = (
    matched_totals["Production_ttpa"] / matched_totals[capacity_col]
).unstack()
print(utilization_matched.round(3))


Technology_type  BF-BOF  EAF
Year                        
2020               0.81 0.92
2021               0.72 0.64
2022               0.81 0.85
2023               0.77 0.69
2024               0.80 0.75


### Q6 [QUANT + QUAL] Naive or matched?

(a) For **BF-BOF in 2023**, report the Part A (naive) utilisation, the matched utilisation, and the official-implied utilisation (from `check`). (b) Which would you use for projections? In one sentence, say what bias could **remain** in the matched estimate (hint: which plants report their production?).

**Your answer (Q6):**

For BF-BOF in 2023, naive utilisation is **7.6%**, matched utilisation is **76.5%**, and official-implied utilisation is **74.7%**. I would use the matched rate for projections because production and capacity refer to the same plants. It could still be biased if plants that report production are more productive than those that do not.

## B3. A range from a sample: the bootstrap

We have a matched utilisation for BF-BOF, but it comes from a **sample** of plants. A different sample would give a slightly different value. How different? The **bootstrap** answers it in four steps:

1. Draw plants **with replacement** from your sample, as many as the sample contains.
2. Recompute the average utilisation (sum of production / sum of capacity).
3. Repeat **1,000 times**.
4. The 5th and 95th percentiles of the 1,000 values form a **90% interval**.

**Goal.** Write a function `boot_mean_u(df, n=1000)` that returns the `n` bootstrap values. Use it to create `ci_all` (interval for **all** matched BF-BOF plants) and `ci_small` (interval for **10 randomly chosen** BF-BOF plants).

> **Hint.** Sum production and capacity **per plant** first (`groupby("Plant ID")`), then resample plant numbers with `np.random.default_rng(42).integers(0, n_plants, size=(n, n_plants))`; the values of one draw are `prod[idx].sum(1) / cap[idx].sum(1)`. `np.percentile(values, [5, 95])`.

In [13]:
def boot_mean_u(df, n=1000):
    by_plant = df.groupby("Plant ID")[["Production_ttpa", capacity_col]].sum()
    prod = by_plant["Production_ttpa"].to_numpy()
    cap = by_plant[capacity_col].to_numpy()
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(by_plant), size=(n, len(by_plant)))
    return prod[idx].sum(axis=1) / cap[idx].sum(axis=1)

bf_matched = obs_matched[obs_matched["Technology_type"] == "BF-BOF"]
small_ids = np.random.default_rng(SEED).choice(
    bf_matched["Plant ID"].unique(), size=10, replace=False
)
small_matched = bf_matched[bf_matched["Plant ID"].isin(small_ids)]
ci_all = np.percentile(boot_mean_u(bf_matched), [5, 95])
ci_small = np.percentile(boot_mean_u(small_matched), [5, 95])
print("All:", ci_all, "width:", ci_all[1] - ci_all[0])
print("Ten:", ci_small, "width:", ci_small[1] - ci_small[0])


All: [0.75369033 0.82189715] width: 0.06820681958410812
Ten: [0.6474075  0.90153138] width: 0.2541238852101261


### Q7 [QUANT + QUAL] Small sample, wide interval

(a) Report `ci_all` and `ci_small` and their widths. (b) Why is the second wider? (c) In one sentence, what does "90% interval" mean here? (d) Which kind of uncertainty (A or E) shrinks when you have more plants?

**Your answer (Q7):**

The 90% interval is **75.4%–82.2%** for all matched BF-BOF plants, a width of **6.8 percentage points**. For 10 plants it is **64.7%–90.2%**, a width of **25.4 points**. The smaller sample gives a less precise estimate. Under the sampling assumptions, this interval method would cover the population mean about 90% of the time; more plants reduce this epistemic uncertainty.

> ### [OPTIONAL] Bayesian version of the same interval
>
> A **credible interval** answers a more direct question: *given the data and what I believed before, where is the true average?* We start from a **prior guess** (mean $\mu_0$, worth $\kappa_0$ observations) and update it with the data (mean $\bar x$, $n$ observations):
>
> posterior mean $=\dfrac{\kappa_0\,\mu_0 + n\,\bar x}{\kappa_0+n}$ (a weighted average: the prior counts as $\kappa_0$ observations).
>
> **Task.** For the **10 plants** of `ci_small`, compute the posterior mean for (i) a **weak** prior ($\mu_0=0.5$, $\kappa_0=0.01$) and (ii) a **wrong** prior ($\mu_0=0.3$, $\kappa_0=5$). The width of a 90% interval is roughly $1.8\times s/\sqrt{n+\kappa_0}$ ($s$ = standard deviation of the 10 values).
> **Question Q7b.** How much does the wrong prior move the answer, and why does this matter when data are scarce?

In [14]:
small_totals = small_matched.groupby("Plant ID")[
    ["Production_ttpa", capacity_col]
].sum()
small_u = small_totals["Production_ttpa"] / small_totals[capacity_col]
sample_mean = small_u.mean()
sample_sd = small_u.std()
weak_mean = (0.01 * 0.5 + len(small_u) * sample_mean) / (0.01 + len(small_u))
wrong_mean = (5 * 0.3 + len(small_u) * sample_mean) / (5 + len(small_u))
weak_width = 1.8 * sample_sd / np.sqrt(len(small_u) + 0.01)
wrong_width = 1.8 * sample_sd / np.sqrt(len(small_u) + 5)
print("Sample mean:", sample_mean)
print("Weak prior:", weak_mean, "approx. width:", weak_width)
print("Wrong prior:", wrong_mean, "approx. width:", wrong_width)


Sample mean: 0.8348197580871675
Weak prior: 0.8344852728143533 approx. width: 0.17923504555608594
Wrong prior: 0.656546505391445 approx. width: 0.14641795599594945


**Your answer (Q7b (optional)):**

The 10-plant sample mean is **83.5%**. The weak prior gives **83.4%** (approximate interval width 17.9 points), while the wrong prior gives **65.7%** (width 14.6 points). The wrong prior pulls the estimate down by **17.8 percentage points** compared with the weak prior, showing how much prior assumptions can matter with little data.

## B4. Simulation: a range for emissions

**Idea (Monte Carlo).** Instead of one value per input, use a *plausible range*, then **repeat the whole calculation many times**, each time with different plausible values. The cloud of results shows the uncertainty of the output.

Two uncertain inputs:

| Input | Plausible range | How to draw |
|---|---|---|
| **Utilisation** per technology | the bootstrap values of B3 | one bootstrap value per draw and technology (all technologies pooled if a technology has fewer than 5 plants) |
| **Emission factor** per technology | multiplier on your Part A factor, **assumed illustrative values** below | triangular distribution (low, mode = 1, high): `rng.triangular(low, 1, high, size=n)` |

| Technology | low | mode | high |
|---|---|---|---|
| BF-BOF | 0.85 | 1 | 1.20 |
| EAF | 0.65 | 1 | 2.00 |
| Other | 0.65 | 1 | 1.40 |

**Goal.** Create `ef_mult_ranges` (dict) and a function `simulate(n=1000, year=2030, vary_util=True, vary_ef=True, bf_cap_mult=1.0, util_mult=1.0, seed=0)` returning a dict with
* `national`: array of `n` national emissions (**MtCO2**) for `year`;
* `national_prod`: array of `n` national productions (**Mt**);
* `by_company`: DataFrame (`n` rows, one column per company, MtCO2).

> **Hint.**
> 1. Take the rows of `china_operational` for `year`. For every plant you need: capacity (missing = 0), technology, your Part A emission factor, company.
> 2. Draw a matrix `u` (n x 3) of utilisations and a matrix `mult` (n x 3) of multipliers (columns = BF-BOF, EAF, Other). If `vary_util=False`, use the pooled matched utilisation of each technology instead (sum of production / sum of capacity over all years); if `vary_ef=False`, use multipliers equal to 1.
> 3. Choose each plant's column by its technology: `u[:, tech_index_of_plant]` gives an (n x plants) matrix. Then production = capacity x utilisation (x `util_mult`), emissions = production x 1,000 x factor x multiplier / 1e6.
> 4. `bf_cap_mult` multiplies the capacity of BF-BOF plants (used for scenarios later).
> 5. Company totals: `pd.DataFrame(em_matrix, columns=company_of_each_plant).T.groupby(level=0).sum().T`.

In [15]:
ef_mult_ranges = {
    "BF-BOF": (0.85, 1.20),
    "EAF": (0.65, 2.00),
    "Other": (0.65, 1.40)
}
technologies = ["BF-BOF", "EAF", "Other"]

def simulate(n=1000, year=2030, vary_util=True, vary_ef=True,
             bf_cap_mult=1.0, util_mult=1.0, seed=0):
    plants = china_operational[china_operational["Year"] == year].copy()
    rng = np.random.default_rng(seed)
    u = np.zeros((n, len(technologies)))
    mult = np.ones((n, len(technologies)))

    for j, technology in enumerate(technologies):
        sample = obs_matched[obs_matched["Technology_type"] == technology]
        if sample["Plant ID"].nunique() < 5:
            sample = obs_matched
        if vary_util:
            u[:, j] = rng.choice(boot_mean_u(sample, n), size=n)
        else:
            u[:, j] = sample["Production_ttpa"].sum() / sample[capacity_col].sum()
        if vary_ef:
            low, high = ef_mult_ranges[technology]
            mult[:, j] = rng.triangular(low, 1, high, size=n)

    tech_index = plants["Technology_type"].map(
        {name: j for j, name in enumerate(technologies)}
    ).to_numpy()
    capacity = plants[capacity_col].fillna(0).to_numpy(dtype=float, copy=True)
    capacity[plants["Technology_type"].eq("BF-BOF").to_numpy()] *= bf_cap_mult
    production = capacity[None, :] * u[:, tech_index] * util_mult
    emissions = production * plants[
        "Emission_factor_tCO2_per_t"
    ].to_numpy()[None, :] * mult[:, tech_index] / 1000
    by_company = pd.DataFrame(
        emissions, columns=plants["Company"].fillna("Unknown")
    ).T.groupby(level=0).sum().T
    return {
        "national": emissions.sum(axis=1),
        "national_prod": production.sum(axis=1) / 1000,
        "by_company": by_company
    }

print("Simulation ready")


Simulation ready


### Q8 [QUANT + QUAL] Read the result

(a) For **2030**, report the **median** and the **90% interval** (5th to 95th percentile) of national emissions, and the **relative half-width** = (upper - lower) / (2 x median). (b) Compare the median with your **Part A value for 2030** (sum of `company_emissions` for 2030, in Mt). Which one is closer to the simulation, and why do they differ so much (hint: B1)?

In [16]:
baseline = simulate()
median_2030 = np.median(baseline["national"])
ci_2030 = np.percentile(baseline["national"], [5, 95])
half_width_2030 = (ci_2030[1] - ci_2030[0]) / (2 * median_2030)
part_a_2030 = company_emissions.loc[
    company_emissions["Year"] == 2030, "estimated_emissions_tCO2"
].sum() / 1e6
print("Simulation (MtCO2):", median_2030, ci_2030,
      "relative half-width:", half_width_2030)
print("Part A (MtCO2):", part_a_2030)


Simulation (MtCO2): 1988.1878321057166 [1762.99451798 2227.82612423] relative half-width: 0.1168983128120945
Part A (MtCO2): 150.597024754866


**Your answer (Q8):**

The 2030 simulation median is **1,988 MtCO₂**, with a 90% interval of **1,763–2,228 MtCO₂** and a relative half-width of **11.7%**. Part A gives only **150.6 MtCO₂** for 2030. The simulation is much higher because it applies utilisation estimated from plants with both production and capacity data; Part A's rate is pulled down by missing production reports, as shown in B1.

### Q9 [QUANT + QUAL] Which input matters most?

Run `simulate` twice: once varying **only the utilisation** (`vary_ef=False`) and once varying **only the emission factor** (`vary_util=False`). For each, give the relative half-width for 2030. Which input creates more of the uncertainty? Could **more data** reduce it?

In [17]:
util_only = simulate(vary_ef=False)
ef_only = simulate(vary_util=False)

def relative_half_width(values):
    lower, upper = np.percentile(values, [5, 95])
    return (upper - lower) / (2 * np.median(values))

print("Utilisation only:", relative_half_width(util_only["national"]))
print("Emission factor only:", relative_half_width(ef_only["national"]))


Utilisation only: 0.04109705245502512
Emission factor only: 0.11078363835584175


**Your answer (Q9):**

The relative half-width is **4.1%** when only utilisation varies and **11.1%** when only emission factors vary. The assumed emission-factor range contributes more uncertainty here. More plant-level emission measurements could narrow that epistemic uncertainty.

### Q10 [QUANT + QUAL] Companies: can you say who is number one?

Using `by_company` of your 2030 run: (a) for the **top 3 companies by median**, give the median and the 90% interval; (b) compute the probability that the #1 company emits **more than** the #2 (fraction of simulations); (c) what would you write in a report instead of "Company X is the largest emitter"?

In [18]:
company_draws = baseline["by_company"]
top_three = company_draws.median().nlargest(3).index
for name in top_three:
    values = company_draws[name]
    print(name, "median:", np.median(values),
          "90% interval:", np.percentile(values, [5, 95]))
leader_probability = (company_draws[top_three[0]] > company_draws[top_three[1]]).mean()
print("P(first > second):", leader_probability)


Angang Steel Co Ltd [100.0%] median: 54.75661448147551 90% interval: [48.11517264 61.56918064]
Baoshan Iron & Steel Co Ltd [100.0%] median: 52.133086988626225 90% interval: [45.80985338 58.61924593]
Beijing Shougang Co Ltd [100.0%] median: 37.65518500254774 90% interval: [33.08797931 42.34006995]
P(first > second): 1.0


**Your answer (Q10):**

| Company | Median (MtCO₂) | 90% interval (MtCO₂) |
|---|---:|---:|
| Angang Steel Co Ltd | 54.8 | 48.1–61.6 |
| Baoshan Iron & Steel Co Ltd | 52.1 | 45.8–58.6 |
| Beijing Shougang Co Ltd | 37.7 | 33.1–42.3 |

Angang exceeds Baoshan in **100% of the 1,000 simulations**. I would report that Angang ranks first under this model's assumptions, while the ownership data and model uncertainty still limit a firm ranking.

## B5. Is the interval honest? Test it

A 90% interval should contain the true value in about 90% of cases. We have one case where the truth is known: **official 2023 production**.

### Q11 [QUANT + QUAL] Does the official value fall inside?

Run `simulate(1000, year=2023)` and look at `national_prod` (Mt). (a) Give its 90% interval and say whether the official 2023 value (1,020 Mt) falls inside. (b) If not, what does this tell you about the interval, **even though it was computed correctly**? (c) What would you do about it?

In [19]:
simulation_2023 = simulate(year=2023)
prod_ci_2023 = np.percentile(simulation_2023["national_prod"], [5, 95])
official_2023_mt = official_kt[2023] / 1000
print("2023 production interval (Mt):", prod_ci_2023)
print("Official value inside:", prod_ci_2023[0] <= official_2023_mt <= prod_ci_2023[1])


2023 production interval (Mt): [1026.04403702 1114.09631122]
Official value inside: False


**Your answer (Q11):**

The simulated 90% interval for 2023 production is **1,026–1,114 Mt**, so the official **1,020 Mt** is just outside it. A correctly calculated interval can still miss when the model or reporting sample is biased. I would check which plants report production and calibrate the model against official totals before using its intervals.

## B6. Scenarios

A **scenario** is a "what if" story. We do **not** attach a probability to it; we run each one through the same simulation and compare.

| Scenario | Story | How |
|---|---|---|
| **S0** | no change | as is |
| **S1** | **30% of BF-BOF capacity is retired** (by 2027, still true in 2030) | `bf_cap_mult=0.7` |
| **S2** | **steel demand drops**: utilisation 10% lower | `util_mult=0.9` |

### Q12 [QUANT + QUAL] Scenario gap vs interval width

(a) Run the three scenarios for 2030 and give the **median and 90% interval** of each. (b) Compare the **gap between** scenario medians with the **width of the interval** within a scenario. Which is bigger? (c) Why would it be misleading to average the three scenarios into one probability-weighted number?

In [20]:
scenarios = {
    "S0": baseline,
    "S1": simulate(bf_cap_mult=0.7),
    "S2": simulate(util_mult=0.9)
}
for name, result in scenarios.items():
    print(name, "median:", np.median(result["national"]),
          "90% interval:", np.percentile(result["national"], [5, 95]))


S0 median: 1988.1878321057166 90% interval: [1762.99451798 2227.82612423]
S1 median: 1419.5506117141886 90% interval: [1260.6428662  1590.82395354]
S2 median: 1789.369048895146 90% interval: [1586.69506618 2005.04351181]


**Your answer (Q12):**

| Scenario | Median (MtCO₂) | 90% interval (MtCO₂) |
|---|---:|---:|
| S0: no change | 1,988 | 1,763–2,228 |
| S1: 30% BF-BOF retirement | 1,420 | 1,261–1,591 |
| S2: 10% lower utilisation | 1,789 | 1,587–2,005 |

S1 is about **569 MtCO₂** below S0, more than S0's **465 MtCO₂** interval width. S2 is about **199 MtCO₂** below S0, less than that width. Averaging the scenarios would imply probabilities for future choices that we have not justified.

## B7. Communicate

Words used by the IPCC for probabilities: **virtually certain** 99-100%, **very likely** 90-100%, **likely** 66-100%, **about as likely as not** 33-66%, **unlikely** 10-33%, **very unlikely** 1-10%, **exceptionally unlikely** 0-1%.

### Q13 [QUANT + QUAL] A probability, a word, and a note

(a) For each scenario, compute the probability that **2030 emissions exceed the 2023 median** (from your `simulate(..., year=2023)`) and give the matching word. (b) Write a note of **about 80 words** for a bank's risk committee about China's steel emissions in 2030. Include: the **median and 90% interval** (baseline), **one sentence on the scenarios**, the **biggest source of uncertainty**, **one limitation** of the model and **one recommended action**. Plain language, no jargon.

> **Hint.** `(draws > threshold).mean()`.

In [21]:
threshold_2023 = np.median(simulation_2023["national"])

def probability_word(p):
    if p >= 0.99:
        return "virtually certain"
    if p >= 0.90:
        return "very likely"
    if p >= 0.66:
        return "likely"
    if p >= 0.33:
        return "about as likely as not"
    if p >= 0.10:
        return "unlikely"
    if p >= 0.01:
        return "very unlikely"
    return "exceptionally unlikely"

print("2023 median (MtCO2):", threshold_2023)
for name, result in scenarios.items():
    p = (result["national"] > threshold_2023).mean()
    print(name, p, probability_word(p))


2023 median (MtCO2): 2000.8928327705046
S0 0.472 about as likely as not
S1 0.0 exceptionally unlikely
S2 0.054 very unlikely


**Your answer (Q13):**

| Scenario | Probability of exceeding the 2023 median | IPCC wording |
|---|---:|---|
| S0 (baseline) | 47.2% | About as likely as not |
| S1 (BF-BOF retirement) | 0.0% | Exceptionally unlikely |
| S2 (lower demand) | 5.4% | Very unlikely |

**Note for the risk committee.** Under the baseline, China’s 2030 steel emissions have a median of **1,988 MtCO₂** and a 90% interval of **1,763–2,228 MtCO₂**. Retiring 30% of BF-BOF capacity lowers the median to 1,420 MtCO₂; a 10% demand reduction lowers it to 1,789 MtCO₂. The assumed emission factors create the widest uncertainty. Production reports cover only a small share of plants, and the model misses the official 2023 production figure. We recommend improving plant coverage and checking emission factors before using these estimates for lending decisions.